# Huấn luyện PhoBERT-base-v2 cho địa chỉ Việt Nam
Chỉ sử dụng train và dev; không sử dụng test 400. Chạy từng ô và giám sát file status trong Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
D = Path('/content/drive/MyDrive/PhoBERT_Address_Colab_20261030')
ARCHIVE = D / 'PhoBERT_Colab_v1_20261030.zip'
assert ARCHIVE.is_file(), f'Archive not found: {ARCHIVE}'
print('DRIVE_PACKAGE:', ARCHIVE)


In [ ]:
import zipfile
with zipfile.ZipFile(ARCHIVE) as z:
    assert z.testzip() is None
    assert all('test' not in Path(p).name.lower() for p in z.namelist() if p.endswith('.jsonl'))
    z.extractall('/content')
P = Path('/content/PhoBERT_Colab_v1_20261030')
print('UNPACKED:', P)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(P / 'requirements.txt')], check=True)
print('INSTALL_PASS')


In [ ]:
subprocess.run([sys.executable, str(P / 'check_data.py'), '--data-dir', str(P / 'data'), '--manifest', str(P / 'package_manifest.json')], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Please select Colab GPU runtime (T4 or better)'
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)


## Smoke test, 8 train + 4 dev rows
Phải chạy thành công và có summary.json trước khi chạy full training.

In [ ]:
from datetime import datetime, timezone
def fresh_output(prefix):
    path = D / 'runs' / (prefix + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    assert not path.exists()
    return path
SMOKE = fresh_output('smoke')
cmd = [sys.executable, str(P / 'train.py'), '--data-dir', str(P / 'data'), '--manifest', str(P / 'package_manifest.json'), '--out-dir', str(SMOKE), '--smoke', '--epochs', '1', '--batch-size', '4']
subprocess.run(cmd, check=True)
assert (SMOKE / 'summary.json').is_file(), 'Smoke failed - STOP'
print('SMOKE_PASS:', SMOKE)


## Huấn luyện đầy đủ
Theo dõi Google Drive/runs/<run-id>/status.json và history.json sau mỗi epoch.

In [ ]:
RUN = fresh_output('phobert_base_v2')
cmd = [sys.executable, str(P / 'train.py'), '--data-dir', str(P / 'data'), '--manifest', str(P / 'package_manifest.json'), '--out-dir', str(RUN), '--epochs', '6', '--batch-size', '8', '--grad-accum', '2', '--seed', '20261030', '--lr', '2e-5', '--patience', '2']
print('LIVE_MONITOR:', RUN / 'status.json')
subprocess.run(cmd, check=True)
assert (RUN / 'summary.json').is_file()
print('TRAINING_COMPLETE:', RUN)


In [ ]:
import json
print(json.dumps(json.loads((RUN / 'summary.json').read_text()), indent=2, ensure_ascii=False))
print('BEST_MODEL:', RUN / 'best_model')
print('Note: token BIO metrics on weak labels, not end-to-end six-field accuracy.')
